# 10 · Training: undefended predictor R0 and the one-shot baselines

* **R0** (`r0_s0..2`): SSL-MOS (wav2vec 2.0 base + mean-pooled regression head) trained on BVCC, 3 seeds.
  About 20 min on one A100 each.
* **Baselines** (`baseline_{noise,pgd,random}_s0..2`): R0 fine-tuned once on preference pairs made by additive noise,
  waveform PGD, or random codec-latent directions, with the same step budget as three patch rounds.
  They start from the **released** R0 of the same seed, so they can be retrained without retraining R0.

Outputs go to `{scratch}/projects/audio_assessment_hackability/retrain/<key>/`.

In [1]:
import os, sys
from pathlib import Path

# Works when started from the repo root or from notebooks/; needs the project environment
# (`source scripts/env.sh`, kernel "hackpatch" — see 00_setup.ipynb).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first key); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['r0', 'baseline']]
print(len(KEYS), "runs:", ", ".join(KEYS))

12 runs: r0_s0, r0_s1, r0_s2, baseline_noise_s0, baseline_random_s0, baseline_pgd_s0, baseline_noise_s1, baseline_random_s1, baseline_pgd_s1, baseline_noise_s2, baseline_random_s2, baseline_pgd_s2


In [4]:
# Resolved configuration of the first run (checkpoint paths already point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: train_r0
task: train_predictor
phase: main
seed: 0
run_name: train_r0_s0__2026-09-15_15-31-19
init_ckpt: null
model:
  pretrained: facebook/wav2vec2-base
  freeze_feature_encoder: true
  freeze_encoder: false
data:
  corpus: bvcc
  max_seconds: 6.0
  num_workers: 8
train:
  pair_source: none
  steps: 8000
  batch_size: 8
  pair_batch_size: 8
  lr_encoder: 2.0e-05
  lr_head: 0.001
  weight_decay: 0.01
  warmup_steps: 500
  cosine: true
  grad_clip: 1.0
  w_mse: 1.0
  w_rank: 0.0
  margin: 0.5
  noise_snr_db:
  - 5.0
  - 30.0
  pgd:
    eps: 0.002
    steps: 5
    step_size: 0.0005
  log_every: 25
  eval_every: 500
  select: best
  pairs_csv: null
  val_pairs_csv: null
eval:
  batch_size: 16
  max_seconds: 20.0
  n_boot: 1000
wandb:
  entity: alialavi
  mode: online
  group: null



In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for r0_s0 ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_r0_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key r0_s0



## Check a retrained predictor

Agreement with human ratings on BVCC test for every finished retrained predictor, next to the paper's value for the
same run (`analysis/out/e2_long.csv`). Needs a GPU in this kernel (about a minute per predictor).

In [6]:
import torch
from hackpatch.data.manifests import read_manifest
from hackpatch.eval.correlation import correlation_report
from hackpatch.models.ssl_mos import load_predictor
from hackpatch.patch.trainer import predict_manifest

paper = pd.read_csv(ROOT / "analysis/out/e2_long.csv").query("corpus == 'bvcc' and split == 'test'")
name = {"r0": "R0", "baseline": None}
rows = []
if torch.cuda.is_available():
    test = read_manifest("bvcc", "test")
    for k in KEYS:
        sel = repro.RUNS / "retrain" / k / "results" / "selected_checkpoint.txt"
        if not sel.exists():
            continue
        seed = int(k.rsplit("_s", 1)[1])
        pred_name = "R0" if k.startswith("r0") else k.rsplit("_s", 1)[0]
        df = test.copy()
        df["pred"] = predict_manifest(load_predictor(sel.read_text().strip()), df, 16, 20.0)
        rep = correlation_report(df, n_boot=100, seed=seed)
        ref = paper[(paper.predictor == pred_name) & (paper.seed == seed)]
        rows.append({"key": k, "sys_srcc": rep["sys_srcc"], "paper_sys_srcc": ref.sys_srcc.mean(),
                     "utt_mse": rep["utt_mse"], "paper_utt_mse": ref.utt_mse.mean()})
pd.DataFrame(rows) if rows else "no GPU in this kernel or no finished retrained predictor"

'no GPU in this kernel or no finished retrained predictor'